<img src="https://industrial.uniandes.edu.co/sites/default/files/imagenes/uniandeslogo.png" alt="Universidad de los Andes" style="float: right; width: 300px; height: auto;">

# imputacion_NTL_Colombia

Editor: Juan Diego Heredia - jd.heredian@uniandes.edu.co

Marzo 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `glob` y `unicodedata`: para lectura de múltiples archivos y normalización de texto.
- `warnings`: para suprimir advertencias irrelevantes durante la ejecución.
- `yaml` y `pathlib`: para lectura de rutas desde archivo de configuración.

In [1]:
import pandas as pd
import numpy as np
import glob
import unicodedata
import warnings
import yaml
from pathlib import Path

### Configuración de rutas

Carga los directorios principales desde el archivo de configuración `paths.yml` para mantener rutas organizadas y portables entre sistemas.

In [2]:
with open('paths.yml', 'r') as file:
    paths = yaml.safe_load(file)

raw       = Path(paths['data']['raw'])
temp      = Path(paths['data']['temp'])
processed = Path(paths['data']['processed'])

### Configuración de opciones y estilo

Establece configuraciones que aplican a todo el notebook:
- Supresión de advertencias

In [3]:
warnings.filterwarnings('ignore')

### Carga de archivos NTL

Lee los 13 archivos CSV anuales descargados de Google Earth Engine y los concatena en un único dataframe. Cada archivo corresponde a un año de la serie VIIRS para los municipios de Colombia.

In [4]:
carpeta = raw / 'ntl'

archivos = sorted(carpeta.glob('NTL_VIIRS_CO_*.csv'))

df_ntl_0 = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)

In [5]:
df_ntl_muns = pd.read_excel(raw/'ntl'/'cod_mun'/'muns_ntl_divipola.xlsx').dropna(subset=['cod_divipola'])
df_ntl_muns['mun_code'] = df_ntl_muns['cod_divipola'].astype(int).astype(str).str.zfill(5)

### Normalización de nombres

Genera versiones normalizadas de `municipio` y `departamento` para facilitar el cruce posterior con otras fuentes. La normalización elimina tildes y convierte a minúsculas, dado que GAUL usa nombres con variaciones respecto a DIVIPOLA.

In [6]:
def normalizar_nombre(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize('NFD', str(s))
    s = s.encode('ascii', 'ignore').decode('utf-8')
    return s.lower().strip()

df_ntl_0['municipio_norm']    = df_ntl_0['municipio'].apply(normalizar_nombre)
df_ntl_0['departamento_norm'] = df_ntl_0['departamento'].apply(normalizar_nombre)


#df_ntl = df_ntl.sort_values(['municipio', 'departamento', 'year', 'month']).reset_index(drop=True)

In [7]:
df_ntl = (
    df_ntl_0
    .merge(
        df_ntl_muns[['municipio_norm', 'departamento_norm', 'mun_code']], 
        on=['municipio_norm', 'departamento_norm'], 
        how='left'
        )
    .drop(columns=['municipio', 'departamento','municipio_norm', 'departamento_norm'])
    .sort_values(['mun_code', 'year', 'month']).reset_index(drop=True)
)

df_ntl['dpto_code'] = df_ntl['mun_code'].str[:2]

### Construcción del panel balanceado mensual

Genera el espacio completo municipio × período mediante un cruce cartesiano. GEE omite filas cuando un municipio no tiene píxeles válidos en un mes, por lo que el panel tiene filas faltantes que deben crearse antes de imputar. El resultado garantiza 153 meses por municipio (abril 2012 – diciembre 2024).

**Pasos realizados:**

1. **Catálogo de municipios:** extrae las combinaciones únicas de municipio, departamento y código.
2. **Grilla de períodos:** genera todos los pares año-mes entre abril de 2012 y diciembre de 2024.
3. **Cruce cartesiano:** combina el catálogo con la grilla y une los valores observados.

In [8]:
# Catálogo de municipios únicos
df_municipios = df_ntl[['mun_code', 'adm2_code']]\
    .drop_duplicates(subset=['mun_code'])

# Grilla de períodos año-mes
df_periodos = pd.DataFrame(
    [{'year': y, 'month': m} for y in range(2012, 2026) for m in range(1, 13)
     if not (y == 2012 and m < 4)]
)
df_periodos['period'] = df_periodos['year'].astype(str) + '-' + df_periodos['month'].apply(lambda x: f'{x:02d}')

# Cruce cartesiano y unión con observaciones
df_panel = df_municipios.assign(_k=1)\
    .merge(df_periodos.assign(_k=1), on='_k')\
    .drop(columns='_k')

df_panel = df_panel.merge(
    df_ntl,
    on=['mun_code', 'adm2_code', 'year', 'month', 'period'],
    how='left'
)

### Imputación mensual de valores faltantes

Aplica una estrategia de imputación en cascada sobre las columnas `ntl_mean`, `ntl_median`, `ntl_sum` y `ntl_sd`. La imputación se realiza en el nivel mensual antes de agregar a trimestres, para que los promedios trimestrales no hereden nulos por meses con nubosidad extrema. La marca `ntl_imputado` cubre tanto los nulos como los valores calculados con menos de 10 píxeles válidos (`ntl_flag_ok = 0`).

**Pasos realizados:**

1. **Promedio estacional:** reemplaza con el promedio histórico del municipio para el mismo mes, calculado solo sobre observaciones reales.
2. **Interpolación lineal temporal:** cubre gaps de hasta 6 meses consecutivos dentro de cada municipio.
3. **Mediana departamento-mes:** cubre municipios sin ningún dato propio en toda la serie (casos extremos).

In [9]:
cols_ntl = ['ntl_mean', 'ntl_median', 'ntl_sum', 'ntl_sd']

# Marca de imputación: nulo o flag de baja cobertura de píxeles
df_panel['ntl_imputado'] = (
    df_panel['ntl_mean'].isna() |
    (df_panel['ntl_flag_ok'].fillna(0) == 0)
).astype(int)

# Paso 1: promedio estacional municipio-mes
df_medias_est = (
    df_panel.query('ntl_imputado == 0')
    .groupby(['mun_code', 'month'])[cols_ntl]
    .mean()
    .rename(columns={c: c + '_est' for c in cols_ntl})
    .reset_index()
)
df_panel = df_panel.merge(df_medias_est, on=['mun_code', 'month'], how='left')
for col in cols_ntl:
    mask = df_panel[col].isna()
    df_panel.loc[mask, col] = df_panel.loc[mask, col + '_est']
df_panel.drop(columns=[c + '_est' for c in cols_ntl], inplace=True)

# Paso 2: interpolación lineal temporal (máx. 6 meses)
df_panel = df_panel.sort_values(['mun_code', 'year', 'month'])
for col in cols_ntl:
    df_panel[col] = df_panel.groupby(['mun_code'])[col]\
        .transform(lambda x: x.interpolate(method='linear', limit_direction='both', limit=6))

# Paso 3: mediana departamento-mes para casos sin historial propio
for col in cols_ntl:
    df_panel[col] = df_panel[col].fillna(
        df_panel.groupby(['dpto_code', 'month'])[col].transform('median')
    )

### Agregación mensual a trimestral

Colapsa el panel mensual a trimestres. Para cada municipio y trimestre construye variables de nivel, volatilidad, tendencia y calidad de la observación.

**Variables construidas:**

- **Nivel:** media y mediana trimestral de `ntl_mean`, y suma de `ntl_sum`.
- **Volatilidad:** desviación estándar y coeficiente de variación de `ntl_mean` entre los tres meses.
- **Tendencia intra-trimestral:** diferencia entre el último y el primer mes del trimestre.
- **Estado reciente:** valor del último mes del trimestre sin promediar.
- **Calidad:** número de meses con `ntl_flag_ok = 1` y proporción de meses imputados.

In [10]:
df_panel['quarter'] = pd.to_datetime(
    df_panel['year'].astype(str) + '-' + df_panel['month'].astype(str) + '-01'
).dt.to_period('Q')

grp = df_panel.groupby(['mun_code', 'adm2_code','quarter'])

# Nivel
df_ntl_q = grp['ntl_mean'].agg(ntl_mean_q='mean', ntl_median_q='median').reset_index()
df_ntl_q['ntl_sum_q'] = grp['ntl_sum'].sum().values

# Volatilidad intra-trimestral
df_ntl_q['ntl_sd_q'] = grp['ntl_mean'].std().values
df_ntl_q['ntl_cv_q'] = df_ntl_q['ntl_sd_q'] / (df_ntl_q['ntl_mean_q'] + 1e-8)

# Tendencia intra-trimestral
df_ntl_q['ntl_trend_q'] = grp['ntl_mean'].last().values - grp['ntl_mean'].first().values

# Estado del último mes
df_ntl_q['ntl_last_q'] = grp['ntl_mean'].last().values

# Calidad
df_ntl_q['ntl_meses_validos_q']  = grp['ntl_flag_ok'].sum().values
df_ntl_q['ntl_prop_imputado_q']  = grp['ntl_imputado'].mean().values

### Construcción de variables derivadas trimestrales

Genera features sobre la serie trimestral para que sean coherentes con la unidad de análisis del modelo de violencia.

**Variables construidas:**

- **Log-transformación:** comprime la distribución sesgada a la derecha causada por la diferencia de órdenes de magnitud entre capitales y municipios rurales.
- **Diferencia trimestral:** velocidad de cambio entre trimestres consecutivos.
- **Cambio porcentual trimestral:** variación relativa al trimestre anterior, acotada entre -5 y 5.
- **Z-score con ventana expansiva:** qué tan atípico es el trimestre para el municipio, usando solo historia acumulada hasta `t` para evitar leakage.
- **Media móvil de 2 trimestres:** suaviza ruido residual de nubosidad sin perder la señal de tendencia.

In [11]:
df_ntl_q = df_ntl_q.sort_values(['mun_code', 'quarter'])

# Log-transformación
df_ntl_q['ntl_log_mean_q']   = np.log1p(df_ntl_q['ntl_mean_q'])
df_ntl_q['ntl_log_sum_q']    = np.log1p(df_ntl_q['ntl_sum_q'])
df_ntl_q['ntl_log_median_q'] = np.log1p(df_ntl_q['ntl_median_q'])

# Diferencia trimestral
df_ntl_q['ntl_delta_q'] = df_ntl_q.groupby(['mun_code'])['ntl_mean_q'].diff()

# Cambio porcentual trimestral acotado
df_ntl_q['ntl_pct_change_q'] = df_ntl_q.groupby(
    ['mun_code'])['ntl_mean_q'].pct_change().clip(-5, 5)

# Z-score con ventana expansiva para evitar leakage
def zscore_expanding(x):
    mu  = x.expanding().mean()
    std = x.expanding().std().replace(0, np.nan)
    return (x - mu) / std.fillna(1)

df_ntl_q['ntl_zscore_q'] = df_ntl_q.groupby(
    ['mun_code'])['ntl_mean_q'].transform(zscore_expanding)

# Media móvil de 2 trimestres
df_ntl_q['ntl_ma2q'] = df_ntl_q.groupby(['mun_code'])['ntl_mean_q']\
    .transform(lambda x: x.rolling(2, min_periods=1).mean())

### Resumen del panel trimestral

Imprime estadísticas de cobertura del panel y la tasa de imputación por departamento.

In [12]:
print(f"total filas            : {len(df_ntl_q):,}")
print(f"municipios             : {df_ntl_q['mun_code'].nunique()}")
print(f"trimestres             : {df_ntl_q['quarter'].min()} → {df_ntl_q['quarter'].max()}")
print(f"prop. trimestres imput.: {df_ntl_q['ntl_prop_imputado_q'].mean()*100:.1f}% promedio")
print(f"nulls en ntl_mean_q    : {df_ntl_q['ntl_mean_q'].isna().sum()}")

total filas            : 58,575
municipios             : 1065
trimestres             : 2012Q2 → 2025Q4
prop. trimestres imput.: 11.6% promedio
nulls en ntl_mean_q    : 0


In [13]:
df_ntl_q

,mun_code,adm2_code,quarter,ntl_mean_q,ntl_median_q,ntl_sum_q,ntl_sd_q,ntl_cv_q,ntl_trend_q,ntl_last_q,ntl_meses_validos_q,ntl_prop_imputado_q,ntl_log_mean_q,ntl_log_sum_q,ntl_log_median_q,ntl_delta_q,ntl_pct_change_q,ntl_zscore_q,ntl_ma2q
0,05001,13408,2012Q2,5.899952,2.360983,1082.168912,8.082230,1.369881,-12.787037,2.360983,2,0.333333,1.931514,6.987646,1.212234,NaN,NaN,0.000000,5.899952
1,05001,13408,2012Q3,6.884553,6.794156,1267.650218,0.432439,0.062813,0.560890,7.355045,2,0.333333,2.064906,7.145709,2.053374,0.984601,0.166883,0.707107,6.392253
2,05001,13408,2012Q4,16.732493,15.763193,30015.588324,2.297220,0.137291,3.592306,19.355498,3,0.000000,2.875399,10.309505,2.819186,9.847940,1.430440,1.150794,11.808523
3,05001,13408,2013Q1,15.596042,17.779009,59906.329010,5.137666,0.329421,-1.502745,17.779009,3,0.000000,2.809164,11.000554,2.932740,-1.136451,-0.067919,0.760819,16.164267
4,05001,13408,2013Q2,11.415502,13.703035,19913.189510,4.220985,0.369759,7.158511,13.703035,3,0.000000,2.518946,9.899188,2.688054,-4.180539,-0.268051,0.022338,13.505772
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58570,99773,14414,2024Q4,0.309808,0.312898,60948.487950,0.031014,0.100106,-0.061796,0.277365,3,0.000000,0.269881,11.017801,0.272237,-0.017391,-0.053152,-0.013711,0.318504
58571,99773,14414,2025Q1,0.330374,0.278868,64295.334495,0.121702,0.368377,0.226466,0.469360,3,0.000000,0.285460,11.071258,0.245976,0.020566,0.066381,0.049450,0.320091
58572,99773,14414,2025Q2,0.349901,0.321053,3811.789964,0.077099,0.220344,-0.145878,0.291386,3,0.000000,0.300031,8.246116,0.278429,0.019527,0.059106,0.109432,0.340137
58573,99773,14414,2025Q3,0.254958,0.265422,29480.387332,0.043653,0.171214,0.085403,0.292428,3,0.000000,0.227102,10.291514,0.235405,-0.094943,-0.271342,-0.188321,0.302430


### Exportación del panel

Guarda el panel trimestral en formato parquet en el directorio de datos procesados.

In [14]:
df_ntl_q.to_parquet(temp / 'ntl' /'ntl.parquet', index=False)